In [1]:
import mlflow
import optuna
import mlflow.sklearn
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

dagshub_token = os.getenv("DAGSHUB_PAT")

if not dagshub_token:
    raise EnvironmentError("DAGSHUB_PAT environment variable is not set")

# DagsHub credentials for MLflow
os.environ["MLFLOW_TRACKING_USERNAME"] = "rajeshxdatascience"
os.environ["MLFLOW_TRACKING_PASSWORD"] = dagshub_token

# MLflow tracking URI
repo_owner = "rajeshxdatascience"
repo_name = "yt-comment-sentiment-analysis"

mlflow.set_tracking_uri(
    f"https://dagshub.com/{repo_owner}/{repo_name}.mlflow"
)

In [3]:
df = pd.read_csv(r"C:\Users\rajes\Desktop\yt-comment-sentiment-analysis\data\processed\reddit_preprocessing.csv").dropna(subset=['clean_comment'])
df.head()

,clean_comment,category
0,family mormon never tried explain still stare ...,1
1,buddhism much lot compatible christianity espe...,1
2,seriously say thing first get complex explain ...,-1
3,learned want teach different focus goal not wr...,0
4,benefit may want read living buddha living chr...,1


In [4]:
df.shape

(36662, 2)

In [5]:
mlflow.set_experiment("Exp 6 - LightGBM HP Tuning")

2026/08/13 23:44:23 INFO mlflow.tracking.fluent: Experiment with name 'Exp 6 - LightGBM HP Tuning' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/83e51e4202b24e5fae4b828af513793f', creation_time=1786644864463, effective_trace_archival_retention=None, experiment_id='6', last_update_time=1786644864463, lifecycle_stage='active', name='Exp 6 - LightGBM HP Tuning', tags={}, trace_location=None, workspace='default'>

In [9]:
# Step 1: Remap the class labels from [-1, 0, 1] to [2, 0, 1]
df['category'] = df['category'].map({-1: 2, 0: 0, 1: 1})

# Step 2: Remove rows where the target labels (category) are NaN

df = df.dropna(subset=['category'])

# Step 3: TF-IDF vectorizer setup
ngram_range = (1, 3) # Trigram
max_features = 1000
vectorizer = TfidfVectorizer(ngram_range=ngram_range, max_features=max_features)
X = vectorizer.fit_transform(df['clean_comment'])
y = df['category']

# Step 4: Apply SMOTE to handle class imbalance
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X, y)

# Step 5: Train Test Split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42, stratify=y_resampled)

# Function to log results in MLFLOW
def log_mlflow(model_name, model, X_train, X_test, y_train, y_test, params, trial_number):
    with mlflow.start_run():
        # Log model type
        mlflow.set_tag("mlflow.runName", f"Trial_{trial_number}_{model_name}_SMOTE_TFIDF_Trigrams")
        mlflow.set_tag("experiment_type", "algoritm_comparison")

        # Log algorithm name as a paramter
        mlflow.log_param("algo_name", model_name)

        # Log hyperparameters
        for key, value in params.items():
            mlflow.log_param(key, value)

        # Train model
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
                
        # Log accuracy
        accuracy = accuracy_score(y_test, y_pred)
        mlflow.log_metric("accuracy", accuracy)
                
        # Log Classification report
        classification_rep = classification_report(y_test, y_pred, output_dict=True)
                        
        for label, metrics in classification_rep.items():
            if isinstance(metrics, dict):
                for metric, value in metrics.items():
                     mlflow.log_metric(f"{label}_{metric}", value)
                
        # Log the model
        mlflow.lightgbm.log_model(model,f"{model_name}_model")

# Step 6: Optuna objective function for XGBoost
def objective_lightgbm(trial):
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-1, log=True)
    max_depth = trial.suggest_int('max_depth', 3, 10)
    num_leaves = trial.suggest_int('num_leaves', 20, 150)
    min_child_samples = trial.suggest_int('min_child_samples', 10, 100)
    colsample_bytree = trial.suggest_float('colsample_bytree', 0.5, 1.0)
    subsample = trial.suggest_float('subsample', 0.5, 1.0)
    reg_alpha = trial.suggest_float('reg_alpha', 1e-4, 10.0, log=True) # L1 regularization
    reg_lambda = trial.suggest_float('reg_lambda', 1e-4, 10.0, log=True) # L2 regularization

    params = {
        "n_estimators": n_estimators,
        "learning_rate": learning_rate,
        "max_depth": max_depth,
        "num_leaves": num_leaves,
        "min_child_samples": min_child_samples,
        "colsample_bytree": colsample_bytree,
        "subsample": subsample,
        "subsample_freq": 1,
        "reg_alpha": reg_alpha,
        "reg_lambda": reg_lambda,
        "random_state": 42,
        "verbosity": -1
    }

    model = LGBMClassifier(n_estimators=n_estimators, 
                           learning_rate=learning_rate, 
                           max_depth=max_depth, 
                           num_leaves=num_leaves,
                           min_child_samples=min_child_samples,
                           colsample_bytree=colsample_bytree,
                           subsample=subsample,
                           reg_alpha=reg_alpha,
                           reg_lambda=reg_lambda,
                           random_state=42)

    accuracy = log_mlflow("LightGBM", model, X_train, X_test, y_train, y_test, params, trial.number)
    
    return accuracy

# Step 7: Run Optuna for XGBoost, log the best model only
def run_optuna_experiment():
    study = optuna.create_study(direction="maximize")
    study.optimize(objective_lightgbm, n_trials=100)

    # Get the best parameters and log only the best model
    best_params = study.best_params
    best_model = LGBMClassifier(n_estimators=best_params['n_estimators'], 
                                learning_rate=best_params['learning_rate'], 
                                max_depth=best_params['max_depth'],
                                num_leaves=best_params['num_leaves'],
                                min_child_samples=best_params['min_child_samples'],
                                colsample_bytree=best_params['colsample_bytree'],
                                subsample=best_params['subsample'],
                                reg_alpha=best_params['reg_alpha'],
                                reg_lambda=best_params['reg_lambda'], 
                                random_state=42)

    # Log the best model with MLflow, passing the algo_name as "xgboost"
    log_mlflow("LightGBM", best_model, X_train, X_test, y_train, y_test, best_params, "Best")


# Run the experiment for XGboost
run_optuna_experiment()

[I 2026-08-14 00:31:38,041] A new study created in memory with name: no-name-b81cfb6e-9aa3-40f9-bb21-e5fdabe140a7


[LightGBM] [Info] Number of positive: 12616, number of negative: 12616
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.013881 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 56104
[LightGBM] [Info] Number of data points in the train set: 25232, number of used features: 676
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

2026/08/14 00:32:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Trial_0_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/89990228158c43c084bb1de65d9af8cc
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6


[W 2026-08-14 00:32:52,429] Trial 0 failed with parameters: {'n_estimators': 64, 'learning_rate': 0.0071549916706843434, 'max_depth': 4, 'num_leaves': 100, 'min_child_samples': 99, 'colsample_bytree': 0.5315798025395072, 'subsample': 0.5512587178933106, 'reg_alpha': 0.0014304764011142314, 'reg_lambda': 0.127011161806499} because of the following error: The value None could not be cast to float..
[W 2026-08-14 00:32:52,430] Trial 0 failed with value None.


...

[LightGBM] [Info] Number of positive: 12616, number of negative: 12616
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018454 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 64143
[LightGBM] [Info] Number of data points in the train set: 25232, number of used features: 964
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with p

2026/08/14 03:02:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run Trial_99_LightGBM_SMOTE_TFIDF_Trigrams at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6/runs/995eb562fd93464197d8c0fe584a5538
🧪 View experiment at: https://dagshub.com/rajeshxdatascience/yt-comment-sentiment-analysis.mlflow/#/experiments/6


[W 2026-08-14 03:02:35,163] Trial 99 failed with parameters: {'n_estimators': 148, 'learning_rate': 0.00035838869803137506, 'max_depth': 9, 'num_leaves': 52, 'min_child_samples': 35, 'colsample_bytree': 0.6094135137631449, 'subsample': 0.7311978813225875, 'reg_alpha': 8.727120251134656, 'reg_lambda': 1.449102330390507} because of the following error: The value None could not be cast to float..
[W 2026-08-14 03:02:35,165] Trial 99 failed with value None.


ValueError: No trials are completed yet.